# Imported Libraries

In [ ]:
# # Installed all the required libraries.
# !pip install \
# numpy \
# pandas \
# matplotlib \
# seaborn \
# kagglehub \
# scikit-learn \
# imblearn \
# scipy \
# shap

In [ ]:
# Import all the required libraries for the project.
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

import kagglehub

import os

from pathlib import Path

import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score,
                             roc_curve,
                             auc,
                             matthews_corrcoef,
                             confusion_matrix,
                             classification_report)

from imblearn.under_sampling import RandomUnderSampler

from scipy.stats import randint, uniform

import shap

# Directory Handling

In [ ]:
# Variables pointing to directory locations.
# Purpose is for easy saving and loading of assets.

# Find the base directory.
base_dir = Path.cwd()

# Find root directory.
root_dir = next(parent for parent in base_dir.parents if parent.name == 'Web_Phish_Project')

fig_dir = root_dir/'saved_data'/'figures'
data_res = root_dir/'saved_data'/'data_results'

# Set paths relative to Main directory.
eda_graph_dir = fig_dir/'eda_graphs'
eval_graph_dir = fig_dir/'evaluation_graphs'
shap_graph_dir = fig_dir/'shap_graphs'

eda_data_dir = data_res/'eda'
baseline_data_dir = data_res/'baseline_results'
train_data_dir = data_res/'training_results'
test_data_dir = data_res/'test_results'

saved_models_dir = root_dir/'saved_data'/'trained_models'

datasets_dir = root_dir/'saved_data'/'datasets'

def save_csv(file_name, dir_path, data_frame):

    """
    Save a Pandas data frame object.

    Parameters
    ----------
    file_name : string
        Name of file to be saved. Do not include file extension.
    dir_path : string
        The directory path to save the file.
    data_frame : pandas.DataFrame
        The data frame object to save.
    """

    # Create file path if it does not exist.
    os.makedirs(dir_path, exist_ok = True)

    file_path = os.path.join(dir_path, f'{file_name}.csv')

    # If file already exists skip saving.
    if os.path.exists(file_path):
        print(f'{file_name} already exists at {dir_path}. Continuing.')
    else:
        data_frame.to_csv(file_path)
        print(f'{file_name} saved successfully at {dir_path}')

def save_img(file_name, dir_path):

    """
    Save a graph as a png image.
    
    Parameters
    ----------
    file_name : string
        Name of file to be saved. Do not include file extension.
    dir_path : string
        The directory path to save the file.
    """
    
    # Create file path if it does not exist.
    os.makedirs(dir_path, exist_ok = True)
    
    file_path = os.path.join(dir_path, f'{file_name}.png')

    # If image already exists skip saving.
    if os.path.exists(file_path):
        print(f'{file_name} already exists at {dir_path}. Continuing.')
    else:
        plt.savefig(file_path)
        print(f'{file_name} saved successfully at {dir_path}')

# Acquire Dataset

In [ ]:
# Creators Acknowledgement:
# Hannousse, A. and Yahiouche, S. (2021) ‘Web page phishing detection’,
# Mendeley Data, V3, doi: 10.17632/c2gw7fy2j4.3.

# Dataset accessed via the kagglehub api.
path = kagglehub.dataset_download('shashwatwork/web-page-phishing-detection-dataset')
print(f'Dataset path: {path}')
print(f'Dataset name: {os.listdir(path)}')

df = pd.read_csv(path + '/dataset_phishing.csv')

# Initial EDA

In [ ]:
def print_title(title_string, hyphen_num):
    """
    Print a title with surronding hyphens.

    Parameters
    ----------
    title_string : string
        Name of title to print.
    hyphen_num : int
        Number of hyphens to print on each side of title.
    """
    print('')
    print('-' * hyphen_num + ' ' + title_string + ' ' + '-' * hyphen_num)

In [ ]:
# Check dimensions of dataset.
print(f'Dataframe shape: {df.shape}\n')

# Display first 5 records.
pd.set_option('display.max_columns', None)
display(df.head())

# Display core information, including feature-to-datatype matches.
print_title('CORE DATASET INFORMATION', 10)
print(df.info(), '\n')

# Check for null values.
null_amount = df.isnull().sum().sum()
print(f'Dataset contains {null_amount} null values.')

# Check for duplicates.
dupe_amount = df.duplicated().sum()
print(f'Dataset contains {dupe_amount} duplicates.')

# Check label distribution.
print_title('LABEL COUNT', 6)
print(df['status'].value_counts())

# Display feature groupings for better understanding.
url_features = df.columns[1:57]
print_title('URL EXTRACTED FEATURES', 25)
print(url_features)

content_features = df.columns[57:81]
print_title('CONTENT EXTRACTED FEATURES', 25)
print(content_features)

external_features = df.columns[81:88]
print_title('EXTERNAL EXTRACTED FEATURES', 25)
print(external_features, '\n')

# Make note of statistical report in URL features.

# Check extracted feature groups match the numbers described by authors.
print(f'URL extracted feature count should be 56. Count: {len(url_features)}\n')
print(f'Content extracted feature count should be 24. Count: {len(content_features)}\n')
print(f'External extracted feature count should be 7. Count: {len(external_features)}\n')

# Display statistical information.
display(df.describe())

# Pre-Processing and Detailed EDA

In [ ]:
# No null values and duplicates.
# Feature-to-datatype is consistent for all columns.

# Drop URL name.
try:
  df = df.drop('url', axis = 1)
  print('The url column dropped.\n')
except:
  print('The url column already dropped.\n')
finally:
  display(df.iloc[:, :1].head(1))
  print()

# Convert status string values to integer values.
if not pd.api.types.is_numeric_dtype(df['status']):
  df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1}).astype(int)
  print('Status values converted to integers.\n')
else:
  print('Status values already converted to integers.\n')

display(df['status'].head(3))

# Note: Google index values will be inverted. Currently 0 = indexed and 1 = not indexed.
# The values will be inverted.
if df['google_index'].iloc[0] == 1:
  df['google_index'] = df['google_index'].map({0: 1, 1: 0}).astype(int)
  print('Google index values inverted.\n')
else:
  print('Google index values already inverted.\n')

# Find the top features that influence status.
features_corr_status = df.corr()['status'].drop('status')

# Top features with status correlation regardless of sign (reindex to keep signs).
top_abs_corr = (features_corr_status
                .reindex(features_corr_status
                         .abs()
                         .sort_values(ascending = False)
                         .head(10)
                         .index))

# Top features with positive status correlation (less likely to be phishing).
top_pos_corr = features_corr_status.sort_values(ascending = False).head(10)

# Top features with negative status correlation (more likely to be phishing).
top_neg_corr = features_corr_status.sort_values(ascending = True).head(10)

print_title('TOP 10 ABS STATUS CORRELATION', 3)
display(top_abs_corr)
print_title('TOP 10 POSITIVE STATUS CORRELATION', 3)
display(top_pos_corr)
print_title('TOP 10 NEGATIVE STATUS CORRELATION', 3)
display(top_neg_corr)

# Statistics for top features with absolute status correlation.
abs_labels = top_abs_corr.index.tolist()
df[abs_labels].describe()

In [ ]:
# Create a duplicate df with a column for hue and legend plotting.
df_eda_plot = df.assign(status_label = df['status'].map({0: 'Legit', 1: 'Phish'}))

# Show distribution of the top ten absolute features by status using histograms.
plt.figure(figsize = (8, 20))
for i, feature in enumerate(top_abs_corr.index.tolist()):
  plt.subplot(10, 1, i + 1)
  sns.histplot(data = df_eda_plot,
               x = feature,
               hue = 'status_label',
               palette = {'Legit': 'green', 'Phish': 'red'},
               multiple = 'stack')
  plt.title(f'Distribution of {feature} by status')
  plt.xlabel(feature)
  plt.ylabel('Frequency')

plt.tight_layout()
save_img('eda_barcharts', eda_graph_dir)
plt.show()
plt.close()

In [ ]:
# Show distribution, spread, averages, and outliers of top ten absoloute
# features by status using boxplots.
plt.figure(figsize = (8, 20))
for i, feature in enumerate(top_abs_corr.index.tolist()):
  plt.subplot(5, 2, i + 1)
  sns.boxplot(data = df_eda_plot,
              x = 'status',
              y = feature,
              hue = 'status_label',
              palette = {'Legit': 'green', 'Phish': 'red'},
              legend = False)
  plt.title(f'Boxplot of {feature}')
  plt.xlabel('status')
  plt.ylabel(feature)
  plt.xticks([0, 1], ['Legit', 'Phish'])
plt.tight_layout()
save_img('eda_boxplots', eda_graph_dir)
plt.show()
plt.close()

In [ ]:
# Show clustering of ratio of digits in URL vs number of hyperlinks by status.
# Note: The overall shape and clustering are the focus, not linear realtionships.
plt.figure(figsize = (8, 5))
sns.scatterplot(data = df_eda_plot,
                x = 'ratio_digits_url',
                y = 'nb_hyperlinks',
                hue = 'status_label',
                palette = {'Legit': 'green', 'Phish': 'red'},
                alpha = 0.3)
plt.title(f'Ratio of Digits in URL vs Number of Hyperlinks by Status')
plt.tight_layout()
save_img('eda_cluster', eda_graph_dir)
plt.show()
plt.close()

In [ ]:
# Heatmap of the top ten absolute features and status.
plt.figure(figsize = (10, 8))
abs_corr_matrix = df[top_abs_corr.index.tolist() + ['status']].corr()
sns.heatmap(abs_corr_matrix,
            annot = True,
            fmt = '.2f',
            cmap = 'coolwarm',
            linewidths = 1)
plt.title('Top Ten Absolute Features and Status Correlation Heatmap')
plt.tight_layout()
save_img('eda_heatmap', eda_graph_dir)
plt.show()
plt.close()

In [ ]:
# X and Y label spilt.
x = df.drop('status', axis = 1)
y = df['status']

print(f'Shape of x: {x.shape}.')
print(f'Shape of y: {y.shape}.')

# Stratified train-test split (the default balanced set).
x_train_bal, x_test, y_train_bal, y_test = train_test_split(x,
                                                            y,
                                                            test_size = 0.2,
                                                            random_state = 42,
                                                            stratify = y)
# Check shape of each new set.
print_title('BALANCED SHAPES', 12)
print(f'Shape of balanced x_train: {x_train_bal.shape}.')
print(f'Shape of balanced x_test: {x_test.shape}.')
print(f'Shape of balanced y_train: {y_train_bal.shape}.')
print(f'Shape of balanced y_test: {y_test.shape}.')

# Check stratified/balanced nature of y sets.
print_title('DISTRIBUTION OF BALANCED y_train', 3)
print(y_train_bal.value_counts())
print_title('DISTRIBUTION OF BALANCED y_test', 3)
print(y_test.value_counts())

# Get the number of samples for legitimate records.
num_legit_samples = y_train_bal.value_counts()[0]

# Randomly sample the phishing records by a 10% amount of the number of legitimate samples.
# The result is a 9:1 split of legit:phishing.
under_sample = RandomUnderSampler(sampling_strategy = {1: int(num_legit_samples * 0.1)},
                                  random_state = 42)

x_train_imbal, y_train_imbal = under_sample.fit_resample(x_train_bal, y_train_bal)

# Check shape of each new set.
print_title('IMBALANCED SHAPES', 12)
print(f'Shape of imbalanced x_train: {x_train_imbal.shape}.')
print(f'Shape of imbalanced y_train: {y_train_imbal.shape}.')

# Check imbalanced nature of y set.
print_title('DISTRIBUTION OF IMBALANCED y_train', 3)
print(y_train_imbal.value_counts())

# Train and Test Baseline Models

In [ ]:
# Create baseline model trainer function.
def baseline_trainer(models, x_train, y_train):

  for (model_name, model) in models.items():

    # Train model.
    model.fit(x_train, y_train)
  # End of for loop.

# End of function.

# Create model tester and evaluation store function.
def model_tester(models):

  num_res = {}
  obj_res = {}

  for (model_name, model) in models.items():

    # Get probabilites of positive cases (phishing).
    y_proba = model.predict_proba(x_test)[:, 1]

    # Set the default thresold value lower for imbalanced gradient boosting.
    if (model_name == 'imbal_gb'):
        pred_thresold = 0.2
    else:
        pred_thresold = 0.5
    
    # Convert the positive predictions into integers.
    y_pred = (y_proba >= pred_thresold).astype(int)

    # Calculate FPR, TPR, and thresholds (thresholds will not be stored in dict)
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)


    # Populate results dictionary.
    num_res[model_name] = {'accuracy': accuracy_score(y_test, y_pred),
                           'precision': precision_score(y_test, y_pred, pos_label = 1),
                           'recall': recall_score(y_test, y_pred, pos_label = 1),
                           'f1': f1_score(y_test, y_pred, pos_label = 1),
                           'mcc': matthews_corrcoef(y_test, y_pred),
                           'auc': auc(fpr, tpr),
                           'thresold': pred_thresold}

    obj_res[model_name] = {'cm': confusion_matrix(y_test, y_pred),
                           'fpr': fpr,
                           'tpr': tpr,
                           'cr': classification_report(y_test,
                                                       y_pred,
                                                       target_names = ['Legitimate', 'Phishing'],
                                                       output_dict = True)}

  # End of for loop.

  # Return the dictionary results.
  return num_res, obj_res
# --- End of function ---

# Define baseline models.
bal_baseline_models = {'dt': DecisionTreeClassifier(random_state = 42),
                       'rf': RandomForestClassifier(random_state = 42),
                       'gb': GradientBoostingClassifier(random_state = 42)}

imbal_baseline_models = {'dt': DecisionTreeClassifier(random_state = 42),
                         'rf': RandomForestClassifier(random_state = 42),
                         'gb': GradientBoostingClassifier(random_state = 42)}

# --- BASELINE DEFAULT PARAMTERS ---

# Decision Tree
# - criterion: gini.
# - splitter: best.
# - max_depth: None.
# - min_smaples_split: 2.
# - min_samples_leaf: 1.
# - min_weight_fraction_leaf: 0.0.
# - max_features: None.
# - max_leaf_nodes: None.
# - min_impurity_decrease: 0.0.
# - class_weight: None.
# - ccp_alpha: 0.0.
# - monotonic_cst: None.

# Random Forest
# - n_estimators: 100.
# - criterion: gini.
# - max_depth: None.
# - min_samples_split: 2.
# - min_samples_leaf: 1.
# - min_weight_fraction_leaf: 0.0.
# - max_features: sqrt.
# - max_leaf_nodes: None.
# - min_impurity_decrease: 0.0.
# - bootstrap: True.
# - oob_score: False.
# - n_jobs: None.
# - verbose: 0.
# - warm_start: False.
# - class_weight: None.
# - ccp_alpha: 0.0.
# - max_samples: None.
# - monotonic_cst: None.

# Gradient Boosting
# - loss: log_loss.
# - learning_rate: 0.1.
# - n_estimators: 100.
# - subsample: 1.0.
# - criterion: friedman_mse.
# - min_samples_split: 2.
# - min_samples_leaf: 1.
# - min_weight_fraction_leaf: 0.0.
# - max_depth: 3.
# - min_impurity_decrease: 0.0.
# - int: None.
# - max_features: None.
# - verbose: 0.
# - max_leaf_nodes: None.
# - warm_start: False.
# - validation_fraction: 0.1.
# - n_iter_no_change: None
# - tol: 1e-4.
# - ccp_alpha: 0.0.

In [ ]:
# Call baseline training function on both baseline model dictionaires.
baseline_trainer(bal_baseline_models, x_train_bal, y_train_bal)

baseline_trainer(imbal_baseline_models, x_train_imbal, y_train_imbal)

# Call testing function on both baseline model dictionaires.
bal_baseline_num_res, bal_baseline_obj_res = model_tester(bal_baseline_models)

imbal_baseline_num_res, imbal_baseline_obj_res = model_tester(imbal_baseline_models)

In [ ]:
# Create confusion matrix heatmap function.
def cm_heatmap_printer(models_obj_res, 
                       n_rows, 
                       fig_width, 
                       fig_height, 
                       super_title, 
                       file_name = None, 
                       dir_loc = None):

    class_names = ['Legit', 'Phish']

    plt.figure(figsize = (fig_width, fig_height))
    plt.suptitle(super_title, fontsize = 16)

    for index, (model_name, obj_name) in enumerate(models_obj_res.items()):

        plt.subplot(n_rows, 3, index + 1)

        sns.heatmap(obj_name['cm'],
                    annot = True,
                    fmt = '.0f',
                    cmap = 'Greens',
                    cbar = False,
                    linewidths = 1,
                    linecolor = 'black',
                    xticklabels = class_names,
                    yticklabels = class_names)
  
        plt.title(model_name)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
    # End of for loop.
    
    plt.tight_layout(w_pad = 3)

    if file_name is not None:
        save_img(file_name, dir_loc)

    plt.show()
    plt.close()
# End of function.

# Create ROC AUC graph function.
def roc_auc_graph_printer(models_num_res, 
                          models_obj_res, 
                          graph_title, 
                          file_name = None, 
                          dir_loc = None):
    
    plt.figure(figsize = (6, 4))
    
    for (model_name, obj_name) in models_obj_res.items():
        
        fpr = obj_name['fpr']
        tpr = obj_name['tpr']
        auc_score = models_num_res[model_name]['auc']
        
        plt.plot(fpr, tpr, label = f'{model_name} AUC = {auc_score:.2f}')
    # End of for loop.
    
    plt.plot([0, 1], [0, 1], 'k--')
    
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(graph_title)
    
    plt.legend()
    plt.grid()

    if file_name is not None:
        save_img(file_name, dir_loc)
    
    plt.show()
    plt.close()
# End of function.

# Create classification report printer function.
def cr_printer(models_obj_res, file_name = None, dir_loc = None):

    for (model_name, obj_name) in models_obj_res.items():
    
        print_title(f'{model_name} Classification Report', 10)

        cr_df = pd.DataFrame(obj_name['cr']).T.round(2)
        
        display(cr_df)
        
        if file_name is not None:
            save_name = f'{file_name}_{model_name}'
            save_csv(save_name, dir_loc, cr_df)
    # End of for loop.

# End of function.

In [ ]:
# Create dataframe of numerical results from baseline models.
bal_baseline_df = pd.DataFrame(bal_baseline_num_res).T.round(2)
imbal_baseline_df = pd.DataFrame(imbal_baseline_num_res).T.round(2)

save_csv('bal_baseline_df', baseline_data_dir, bal_baseline_df)
save_csv('imbal_baseline_df', baseline_data_dir, imbal_baseline_df)

# Print numeric metric tables.
print_title('BALANCED BASELINE NUMERIC RESULTS', 10)
display(bal_baseline_df)
print_title('IMBALANCED BASELINE NUMERIC RESULTS', 9)
display(imbal_baseline_df)

# Print confusion matrices.
print_title('BASELINE CONFUSION MATRICES', 37)
cm_heatmap_printer(bal_baseline_obj_res, 
                   1, 8, 3, 
                   'Balanced Baseline Models', 
                   'bal_baseline_heatmap', 
                   eval_graph_dir)
print()
cm_heatmap_printer(imbal_baseline_obj_res,
                   1, 8, 3, 
                   'Imbalanced Baseline Models', 
                   'imbal_baseline_heatmap',
                   eval_graph_dir)

# Print ROC AUC graphs.
print_title('BASELINE ROC AUC GRAPHS', 22)
roc_auc_graph_printer(bal_baseline_num_res,
                      bal_baseline_obj_res,
                      'Balanced Baseline Models',
                      'bal_baseline_roc',
                       eval_graph_dir)
print()
roc_auc_graph_printer(imbal_baseline_num_res,
                      imbal_baseline_obj_res,
                      'Imbalanced Baseline Models',
                      'imbal_baseline_roc',
                      eval_graph_dir)

# Print classification reports.
print_title('BALANCED BASELINE CLASSIFICATION REPORTS', 20)
cr_printer(bal_baseline_obj_res, 'bal_baseline_cr', baseline_data_dir)
print_title('IMBALANCED BASELINE CLASSIFICATION REPORTS', 20)
cr_printer(imbal_baseline_obj_res, 'imbal_baseline_cr', baseline_data_dir)

# Train, Validate, and Tune Models

In [ ]:
# Define models to train and test.
model_dict = {'bal_dt': DecisionTreeClassifier(random_state = 42),
              'imbal_dt': DecisionTreeClassifier(random_state = 42),
              'bal_rf': RandomForestClassifier(random_state = 42),
              'imbal_rf': RandomForestClassifier(random_state = 42),
              'bal_gb': GradientBoostingClassifier(random_state = 42),
              'imbal_gb': GradientBoostingClassifier(random_state = 42)}

# Define ranges of parameters for use in tuning process.
# Note: not all parameters are tuned, only the most important ones.
bal_dt_params = {'criterion': ['gini', 'entropy'],
                 'max_depth': randint(10, 50),
                 'min_samples_split': randint(2, 20),
                 'min_samples_leaf': randint(1, 15),
                 'max_features': [None, 'sqrt', 'log2']}

imbal_dt_params = {'criterion': ['gini', 'entropy'],
                   'max_depth': randint(5, 25),
                   'min_samples_split': randint(25, 100),
                   'min_samples_leaf': randint(15, 50),
                   'max_features': [None, 'sqrt', 'log2'],
                   'class_weight': ['balanced']}

bal_rf_params = {'n_estimators': randint(300, 800),
                 'criterion': ['gini', 'entropy'],
                 'max_depth': randint(10, 60),
                 'min_samples_split': randint(2, 40),
                 'min_samples_leaf': randint(1, 15),
                 'max_features': [None, 'sqrt', 'log2']}

imbal_rf_params = {'n_estimators': randint(300, 800),
                   'criterion': ['gini', 'entropy'],
                   'max_depth': randint(10, 25),
                   'min_samples_split': randint(25, 100),
                   'min_samples_leaf': randint(15, 50),
                   'max_features': [None, 'sqrt', 'log2'],
                   'class_weight': ['balanced', 'balanced_subsample']}

bal_gb_params = {'learning_rate': uniform(0.01, 0.2),
                 'n_estimators': randint(300, 800),
                 'subsample': uniform(0.5, 0.5),
                 'criterion': ['friedman_mse', 'squared_error'],
                 'min_samples_split': randint(2, 40),
                 'min_samples_leaf': randint(1, 20),
                 'max_depth': randint(3, 10)}

imbal_gb_params = {'learning_rate': uniform(0.01, 0.09),
                   'n_estimators': randint(300, 800),
                   'subsample': uniform(0.8, 0.2),
                   'criterion': ['friedman_mse', 'squared_error'],
                   'min_samples_split': randint(25, 100),
                   'min_samples_leaf': randint(15, 50),
                   'max_depth': randint(2, 4)}

# Define stratifed kfold for use in randomised search.
strat_kfold = StratifiedKFold(n_splits = 4, shuffle = True, random_state = 42)

# Random search function for convenience.
def model_tuner(model, params_dict, n_iter_dict):

  # Define random search and its parameters.
  r_search = RandomizedSearchCV(estimator = model,
                                param_distributions = params_dict,
                                n_iter = n_iter_dict,
                                scoring = 'recall',
                                n_jobs = 1,
                                refit = True,
                                cv = strat_kfold,
                                random_state = 42)

  return r_search
# End of function.

# Define model-to-parameter dictionary.
params_dict = {'bal_dt': bal_dt_params,
               'imbal_dt': imbal_dt_params,
               'bal_rf': bal_rf_params,
               'imbal_rf': imbal_rf_params,
               'bal_gb': bal_gb_params,
               'imbal_gb': imbal_gb_params}

# Define number of search iterations dictionary.
n_iter_dict = {'bal_dt': 40,
               'imbal_dt': 30,
               'bal_rf': 50,
               'imbal_rf': 40,
               'bal_gb': 40,
               'imbal_gb': 60}

# Define dictionary to hold best search results.
best_models = {}

# Populate best_models dictionary with parameter tuned models.
for (model_name, model) in model_dict.items():
    best_models[model_name] = model_tuner(model, 
                                          params_dict[model_name], 
                                          n_iter_dict[model_name])
# End of for loop.

In [ ]:
# Folder to hold trained models.
os.makedirs(saved_models_dir, exist_ok = True)

# Train each model on the correct training dataset (balanced or imbalanced).
for (model_name, model) in best_models.items():

    # Create file path from joining folder and model name.
    file_path = os.path.join(saved_models_dir, f'{model_name}.pkl')

    # Check if file path already exists, if it does skip training and saving.
    if os.path.exists(file_path):
        print(f'{model_name}.pkl already exists at {file_path}\nTraining skipped.')
        continue
    
    if model_name.startswith('bal'):
        model.fit(x_train_bal, y_train_bal)
    
    if model_name.startswith('imbal'):
        model.fit(x_train_imbal, y_train_imbal)

    # Save fitted model to path.
    joblib.dump(model, file_path)
    print(f'Model {model_name} trained and saved at {file_path}')
        
# End of for loop.

In [ ]:
# Load each trained model into best_models dictionary.
for model_name in best_models.keys():
    file_path = os.path.join(saved_models_dir, f'{model_name}.pkl')
    best_models[model_name] = joblib.load(file_path)
    print(f'{model_name}.pkl loaded successfully from {file_path}')

In [ ]:
# Return best model parameters, the training score, and mean validation score data frames.
def search_res_df_creator(best_models_dict):

    print_title(f'SEARCH RESULTS', 35)

    # Create lists to hold records.
    best_train_res_list = []
    dt_best_params_list = []
    rf_best_params_list = []
    gb_best_params_list = []
    
    for (model_name, model) in best_models_dict.items():
        
        if model_name.startswith('bal'):
            y_pred = model.predict(x_train_bal)
            recall = recall_score(y_train_bal, y_pred)
            
        if model_name.startswith('imbal'):
            y_pred = model.predict(x_train_imbal)
            recall = recall_score(y_train_imbal, y_pred)

        # Create records containing model name, best params, and best scores.
        best_train_res_record = {'Model': model_name,
                                 'Training Recall': round(recall, 2),
                                 'Mean Validation Recall': round(model.best_score_, 2)}

        best_params_record = {'Model': model_name,
                              'Best Parameters': model.best_params_,}

        # Add the records to the lists.
        best_train_res_list.append(best_train_res_record)
        
        if ('_dt' in model_name):
            dt_best_params_list.append(best_params_record)
        elif ('_rf' in model_name):
            rf_best_params_list.append(best_params_record)
        elif ('_gb' in model_name):
            gb_best_params_list.append(best_params_record)

    # End of for loop.

    # Create data frames from lists.
    best_train_res_df = pd.DataFrame(best_train_res_list)
    dt_best_params_raw = pd.DataFrame(dt_best_params_list)
    rf_best_params_raw = pd.DataFrame(rf_best_params_list)
    gb_best_params_raw = pd.DataFrame(gb_best_params_list)

    # Expand best parameter dictionaries into columns.
    dt_best_params_df = pd.json_normalize(dt_best_params_raw['Best Parameters'])
    dt_best_params_df.insert(0, 'Model', dt_best_params_raw['Model'])
    
    rf_best_params_df = pd.json_normalize(rf_best_params_raw['Best Parameters'])
    rf_best_params_df.insert(0, 'Model', rf_best_params_raw['Model'])

    gb_best_params_df = pd.json_normalize(gb_best_params_raw['Best Parameters'])
    gb_best_params_df.insert(0, 'Model', gb_best_params_raw['Model'])

    # Reset index to remove duplicate index row.
    best_train_res_df = best_train_res_df.set_index('Model')
    dt_best_params_df = dt_best_params_df.set_index('Model')
    rf_best_params_df = rf_best_params_df.set_index('Model')
    gb_best_params_df = gb_best_params_df.set_index('Model')
    
    return best_train_res_df, dt_best_params_df, rf_best_params_df, gb_best_params_df
    
# End of function.

# Call search_res_printer on best models dictionary.
best_train_res_df, dt_best_params_df, rf_best_params_df, gb_best_params_df = search_res_df_creator(best_models)

display(best_train_res_df)
display(dt_best_params_df)
display(rf_best_params_df)
display(gb_best_params_df)

save_csv('best_train_res', train_data_dir, best_train_res_df)
save_csv('dt_best_params_res', train_data_dir, dt_best_params_df)
save_csv('rf_best_params_res', train_data_dir, rf_best_params_df)
save_csv('gb_best_params_res', train_data_dir, gb_best_params_df)

In [ ]:
# Get all the names of features, not including status.
feat_names = df.columns[:-1]

# Dataframe to hold feature importance for each random search best estimator.
best_feat_df = pd.DataFrame({'Feature': feat_names})

# Loop through each best estimator and add their feature importance values to a dataframe and a series.
for (model_name, models) in best_models.items():

    # Add feature importance values to the dataframe.
    best_feat_df[model_name] = models.best_estimator_.feature_importances_
    
    print_title(f'TOP 10 FEATURE IMPORTANCE FOR {model_name}', 3)
    # Create and print the top 10 feature importances for each model.
    best_feat_series = pd.Series(models.best_estimator_.feature_importances_,
                                 index = feat_names).sort_values(ascending = False)
    display(best_feat_series.head(10))

# Find the mean average importance values across each model, sort by largest, and place in dataframe.
top_avg_feat_df = best_feat_df.set_index('Feature').mean(axis = 1).sort_values(ascending = False)

print_title('TOP 10 MEAN FEATURE IMPORTANCE', 3)
display(top_avg_feat_df.head(10))

# Test Models

In [ ]:
# Test the models and get numerical and object results back.
num_res, obj_res = model_tester(best_models)

# Create dataframe of numerical results from best estimators.
res_df = pd.DataFrame(num_res).T.round(2)

# Save numerical results.
save_csv('test_res_df', test_data_dir, res_df)

# Print numeric metric tables.
print_title('NUMERIC TEST SPLIT RESULTS', 9)
display(res_df)

# Print confusion matrices.
print_title('CONFUSION MATRIX TEST SPLIT RESULTS', 32)
cm_heatmap_printer(obj_res,
                   2, 8, 5, 
                   'Confusion Matrix Test Split Results', 
                   'test_res_heatmap',
                   eval_graph_dir)

# Print ROC AUC graphs.
print_title('ROC AUC GRAPH TEST SPLIT RESULTS', 17)
roc_auc_graph_printer(num_res,
                      obj_res,
                      'ROC AUC Test Split Results', 
                      'test_res_roc',
                      eval_graph_dir)

# Print classification reports.
print_title('CLASSIFICATION REPORT TEST SPLIT RESULTS', 6)
cr_printer(obj_res, 'test_res', test_data_dir)

# SHAP Analysis

In [ ]:
# Create a shap dictionary to hold shap object for each model.
shap_dict = {}

# Take a random sample amount from testing data for use in shap.
sampled_x_test = x_test.sample(300, random_state = 42)
sampled_indices = sampled_x_test.index

# Loop through each random search best estimator.
for (model_name, models) in best_models.items():

    # Create explainer object from best estimator.
    explainer = shap.Explainer(models.best_estimator_)

    # Create shap object from explainer and testing data.
    shap_obj = explainer(sampled_x_test)

    # Populate shap dictionary with each model's shap object.
    shap_dict[model_name] = shap_obj
    
# End of for loop.

In [ ]:
# Find original index positions from x_test for mapping to y_test.
original_index = sampled_indices[0]
original_index2 = sampled_indices[1]
print(f'Original index for row 0 is {original_index}.')
print(f'Original index for row 1 is {original_index2}.\n')

# Double check indices match.
print(f'For first sampled row record, label located at row posistion: {y_test.index.get_loc(original_index)}.')
print(f'For second sampled row record, label located at row posistion: {y_test.index.get_loc(original_index2)}.\n')

print(f'Original index for y_test row 1517: {y_test.index[1517]}.')
print(f'Original index for y_test row 1225: {y_test.index[1225]}.\n')

# Find two instances from the sample, one phishing and the other legit.
print('Instance at index 0 belongs to class:', y_test.loc[original_index])
print('Instance at index 1 belongs to class:', y_test.loc[original_index2])

legit_index = 0
phish_index = 1

for (model_name, shap_obj) in shap_dict.items():
    print_title(f'{model_name} SHAP GRAPHS', 45)
    
    if (shap_obj.values.ndim == 3):
        plot_shap_values = shap_obj[:, :, 1]
    else:
        plot_shap_values = shap_obj

    print_title('BEESWARM', 50)
    plt.figure(figsize = (12, 8))
    shap.plots.beeswarm(plot_shap_values, show = False)
    plt.tight_layout()
    save_img(f'{model_name}_beeswarm', shap_graph_dir)
    plt.show()
    plt.close()

    print_title('WATERFALL', 54)
    print('Actual class: Legit')
    plt.figure(figsize = (12, 8))
    shap.plots.waterfall(plot_shap_values[legit_index], show = False)
    plt.tight_layout()
    save_img(f'{model_name}_legit_waterfall', shap_graph_dir)
    plt.show()
    plt.close()
    
    print_title('WATERFALL', 50)
    print('Actual class: Phishing')
    plt.figure(figsize = (12, 8))
    shap.plots.waterfall(plot_shap_values[phish_index], show = False)
    plt.tight_layout()
    save_img(f'{model_name}_phish_waterfall', shap_graph_dir)
    plt.show()
    plt.close()

# Save URL Dataset

In [ ]:
# Features to keep for URL dataset.
keep_features = ['length_url', 'length_hostname', 'ip', 
                 'nb_dots', 'nb_hyphens', 'nb_at',
                 'nb_qm', 'nb_and', 'nb_or', 
                 'nb_eq', 'nb_underscore', 'nb_tilde',
                 'nb_percent', 'nb_slash', 'nb_star', 
                 'nb_colon', 'nb_comma','nb_semicolumn', 
                 'nb_dollar', 'nb_space', 'nb_www', 
                 'nb_com', 'nb_dslash', 'http_in_path', 
                 'https_token', 'ratio_digits_url', 'ratio_digits_host', 
                 'punycode', 'port', 'tld_in_path', 
                 'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains',
                 'prefix_suffix', 'shortening_service', 'path_extension', 
                 'length_words_raw', 'char_repeat', 'shortest_words_raw', 
                 'shortest_word_host', 'shortest_word_path', 'longest_words_raw', 
                 'longest_word_host', 'longest_word_path', 'avg_words_raw', 
                 'avg_word_host', 'avg_word_path', 'phish_hints', 
                 'suspecious_tld', 'status']

# Create new dataset and save for extraction to another notebook.
url_df = df[keep_features].set_index('length_url')
display(url_df.head())
save_csv('url_dataset', datasets_dir, url_df)

In [ ]:
# Notebook end.